# LogicSLM — Colab inference server

Self-contained Flask server for the **critical-reasoning SLM** (Qwen3.5-4B + LoRA). It serves
LogicSLM *and* proxies Claude **Opus 4.8** / **Sonnet 4.6** through your gateway, so the React
frontend can compare all three on one `POST /compare` call — and no secret ever reaches the browser.

**Everything is in this notebook** (nothing to clone or upload). Run the cells top to bottom on a
**GPU** runtime; the last cell prints a public **ngrok URL** — paste it into the frontend's "Server URL"
field.

**Drive layout** (mirrors `build_standalone.py` / `colab_standalone.py`):
- base model `Qwen/Qwen3.5-4B`, LoRA adapter at `/content/drive/MyDrive/SLM/runs/colab/adapter`
- optional secrets file `/content/drive/MyDrive/SLM/.env` (else use Colab Secrets)

Fast wiring check with no GPU: set `MOCK_SLM = True` in the config cell — LogicSLM answers are stubbed
but Opus/Sonnet are still called for real.


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
# MOCK_SLM=True skips loading the 4B (use on a CPU runtime) and returns a stub
# LogicSLM answer while STILL calling the real Opus/Sonnet — a fast wiring check.
MOCK_SLM    = False
BASE_MODEL  = "Qwen/Qwen3.5-4B"
ADAPTER_DIR = "/content/drive/MyDrive/SLM/runs/colab/adapter"   # from colab_standalone.py
DRIVE_ENV   = "/content/drive/MyDrive/SLM/.env"                 # optional secrets file on Drive
PORT        = 5000
OPUS_ID     = "claude-opus-4-8"
SONNET_ID   = "claude-sonnet-4-6"
# Public tunnel. "cloudflare" (default) has NO browser interstitial and passes CORS
# straight through — best for a browser frontend. "ngrok" also works (free tier shows an
# interstitial; the frontend sends `ngrok-skip-browser-warning` to bypass it).
TUNNEL      = "cloudflare"


In [ ]:
# ── Install deps (torch is preinstalled on Colab GPU runtimes) ─────────────
!pip -q install flask flask-cors pyngrok anthropic transformers peft accelerate
!pip install -U "torchao>=0.16.0"

In [ ]:
# ── Mount Drive (holds the adapter + optional .env) ────────────────────────
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ── Secrets: Drive .env first, then Colab Secrets for anything still missing ─
import os
KEYS = ["ANTHROPIC_BASE_URL", "ANTHROPIC_AUTH_TOKEN", "ANTHROPIC_API_KEY", "NGROK_AUTH_TOKEN"]

def load_secrets():
    # 1) KEY=VALUE lines from a .env on Drive (same parser as slm_core._load_dotenv)
    if os.path.exists(DRIVE_ENV):
        for line in open(DRIVE_ENV, encoding="utf-8"):
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            k, v = line.split("=", 1)
            k, v = k.strip(), v.strip().strip('"').strip("'")
            if k and v and not os.environ.get(k):
                os.environ[k] = v
        print("loaded .env from", DRIVE_ENV)
    # 2) Colab Secrets fallback (add them via the key icon in the left sidebar)
    try:
        from google.colab import userdata
        for k in KEYS:
            if not os.environ.get(k):
                try:
                    val = userdata.get(k)
                    if val:
                        os.environ[k] = val
                except Exception:
                    pass
    except Exception:
        pass
    have = [k for k in KEYS if os.environ.get(k)]
    missing = [k for k in KEYS if not os.environ.get(k)]
    print("secrets present:", have)
    if missing:
        print("secrets MISSING:", missing, "-> set them in the Drive .env or Colab Secrets")

load_secrets()


In [ ]:
# ── Inlined from slm_core.py (verbatim) so this notebook is self-contained ──
# Sources: slm_core.py:42-49 (label vocab), :128-138 (frq_prompt),
#          :187-197 (parse_label), :301-330 (_client / call_agent).
import os, re

CANON = {"true": "True", "false": "False",
         "uncertain": "Uncertain", "unknown": "Unknown"}
NEUTRAL = {"Uncertain", "Unknown"}
NEUTRAL_ALIASES = [
    "cannot be determined", "can't be determined", "cannot be known",
    "does not follow", "doesn't follow", "insufficient", "indeterminate",
    "neither", "not enough information", "cannot tell", "can't tell", "neutral",
]

def frq_prompt(it: dict) -> str:
    return (
        "You are an expert in logic and argument analysis. Answer the following "
        "open-ended question about the argument. Reason carefully and commit to "
        "the single best answer - do not hedge by listing many possibilities. Be "
        "specific and concise.\n\n"
        f"{it['stimulus']}\n\n"
        f"Question: {it['question']}\n\n"
        "Explain your reasoning in a few sentences, then clearly state your final "
        "conclusion as exactly one of the labels named in the question."
    )

def parse_label(text, neutral_hint: str = "Unknown"):
    """LAST-hit True/False/Uncertain|Unknown label from natural text (slm_core.py:187)."""
    low = (text or "").lower()
    hits = list(re.finditer(r"\b(true|false|uncertain|unknown)\b", low))
    if hits:
        return CANON[hits[-1].group(1)]
    if any(a in low for a in NEUTRAL_ALIASES):
        return neutral_hint
    return None

_ANTHROPIC_CLIENT = None
def _client():
    """Gateway client from env (ANTHROPIC_BASE_URL + ANTHROPIC_AUTH_TOKEN, Bearer)."""
    global _ANTHROPIC_CLIENT
    if _ANTHROPIC_CLIENT is None:
        import anthropic
        kw = {}
        if os.environ.get("ANTHROPIC_BASE_URL"):
            kw["base_url"] = os.environ["ANTHROPIC_BASE_URL"]
        if os.environ.get("ANTHROPIC_AUTH_TOKEN"):
            kw["auth_token"] = os.environ["ANTHROPIC_AUTH_TOKEN"]   # -> Authorization: Bearer
        elif os.environ.get("ANTHROPIC_API_KEY"):
            kw["api_key"] = os.environ["ANTHROPIC_API_KEY"]
        _ANTHROPIC_CLIENT = anthropic.Anthropic(**kw)
    return _ANTHROPIC_CLIENT

def call_agent(prompt: str, model: str, timeout: int = 60, max_tokens: int = 1024):
    """One Claude call; returns text or None on any failure. (slm_core.py:320; the
    lone addition is printing the error so gateway issues show in the server log.)"""
    try:
        resp = _client().with_options(timeout=float(timeout)).messages.create(
            model=model, max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}])
        text = "".join(b.text for b in resp.content if b.type == "text").strip()
        return text or None
    except Exception as e:
        print(f"call_agent({model}) error:", repr(e))
        return None


In [ ]:
# ── Load LogicSLM: base Qwen3.5-4B (bf16) + LoRA adapter (mirrors colab_standalone.py) ──
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

_device = "cuda" if torch.cuda.is_available() else "cpu"
_dtype  = torch.bfloat16 if _device == "cuda" else torch.float32
tok = None
model = None

if not MOCK_SLM:
    tok = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    try:  # transformers renamed torch_dtype -> dtype; support both
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, trust_remote_code=True, dtype=_dtype,
            attn_implementation="sdpa", device_map=(_device if _device == "cuda" else None))
    except TypeError:
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, trust_remote_code=True, torch_dtype=_dtype,
            attn_implementation="sdpa", device_map=(_device if _device == "cuda" else None))
    if _device != "cuda":
        base = base.to(_device)
    from peft import PeftModel
    model = PeftModel.from_pretrained(base, ADAPTER_DIR)
    model.eval()
    print(f"LogicSLM loaded on {_device}  (adapter: {ADAPTER_DIR})")
else:
    print("MOCK_SLM=True -> skipping model load; LogicSLM answers are stubbed")


In [ ]:
# ── LogicSLM generation: greedy, enable_thinking=False, left-pad (colab_standalone.py) ──
import threading
_slm_lock = threading.Lock()   # serialize GPU use across concurrent requests

def _chat(prompt: str) -> str:
    try:
        return tok.apply_chat_template([{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tok.apply_chat_template([{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def slm_generate(prompt: str, max_new_tokens: int = 512) -> str:
    if MOCK_SLM or model is None:
        return ("(mock) LogicSLM is stubbed on this runtime. The conclusion does not "
                "deductively follow, so the answer is Unknown.")
    with _slm_lock:
        text = _chat(prompt)
        prev = tok.padding_side
        tok.padding_side = "left"
        enc = tok([text], return_tensors="pt", padding=True).to(_device)
        tok.padding_side = prev
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=(tok.pad_token_id or tok.eos_token_id))
        plen = enc["input_ids"].shape[1]
        return tok.decode(out[0][plen:], skip_special_tokens=True)


In [ ]:
# ── Flask API: POST /compare runs LogicSLM + Opus + Sonnet on the SAME prompt ──
import time
from flask import Flask, request, jsonify
from flask_cors import CORS
from concurrent.futures import ThreadPoolExecutor

app = Flask(__name__)
# Allow the browser SPA from any origin + answer CORS preflight for the custom
# `ngrok-skip-browser-warning` header. The frontend only ever talks to this server.
CORS(app, origins="*", allow_headers="*", methods=["GET", "POST", "OPTIONS"])

# The one question all three models answer (labels named inline so parse_label works).
_QUESTION = ("Based only on the premises, is the conclusion True, False, or Unknown "
             "(it does not deductively follow either way)?")

def build_entailment_prompt(premises, conclusion):
    prems = [p.strip() for p in premises if p and p.strip()]
    stimulus = ("Premises:\n" + "\n".join(f"- {p}" for p in prems)
                + f"\n\nConclusion: {conclusion.strip()}")
    return frq_prompt({"stimulus": stimulus, "question": _QUESTION})

def _timed(fn):
    t0 = time.time()
    out = fn()
    return out, int((time.time() - t0) * 1000)

@app.get("/health")
def health():
    return jsonify(ok=True, model_loaded=(model is not None),
                   device=_device, mock=bool(MOCK_SLM))

@app.post("/compare")
def compare():
    data = request.get_json(force=True, silent=True) or {}
    premises = data.get("premises", [])
    if isinstance(premises, str):
        premises = premises.splitlines()
    premises = [p for p in premises if isinstance(p, str) and p.strip()][:20]
    conclusion = (data.get("conclusion") or "").strip()
    if not premises or not conclusion:
        return jsonify(error="Provide at least one premise and a conclusion."), 400
    if len(conclusion) > 2000 or any(len(p) > 2000 for p in premises):
        return jsonify(error="Inputs too long (2000-char cap each)."), 400

    prompt = build_entailment_prompt(premises, conclusion)

    def run_slm():
        raw, ms = _timed(lambda: slm_generate(prompt))
        return {"model": "LogicSLM (Qwen3.5-4B)", "kind": "slm", "ok": True,
                "label": parse_label(raw), "reasoning": raw, "latency_ms": ms}

    def run_frontier(name, kind, mid):
        raw, ms = _timed(lambda: call_agent(prompt, mid, timeout=60, max_tokens=1024))
        if raw is None:
            return {"model": name, "kind": kind, "ok": False, "error": "unavailable",
                    "label": None, "reasoning": None, "latency_ms": ms}
        return {"model": name, "kind": kind, "ok": True,
                "label": parse_label(raw), "reasoning": raw, "latency_ms": ms}

    with ThreadPoolExecutor(max_workers=3) as ex:
        futs = [
            ex.submit(run_slm),
            ex.submit(run_frontier, "Claude Opus 4.8", "opus", OPUS_ID),
            ex.submit(run_frontier, "Claude Sonnet 4.6", "sonnet", SONNET_ID),
        ]
        results = [f.result() for f in futs]   # ordered: LogicSLM, Opus, Sonnet

    return jsonify(prompt_used=prompt, question=_QUESTION, results=results)

print("Flask app ready:  GET /health   POST /compare")


In [ ]:
# ── Open a public tunnel + start the server. Copy the printed https URL into the frontend. ──
# This cell BLOCKS while the server runs — leave it running. Interrupt (stop) to restart.
# TUNNEL="cloudflare" (default): no browser interstitial, clean CORS — best for the SPA.
# TUNNEL="ngrok": works too; the free-tier interstitial is bypassed by the frontend's
#                 `ngrok-skip-browser-warning` header (see the React api.js).
import subprocess, re, stat, urllib.request

def _announce(url):
    print("=" * 72)
    print("  LogicSLM server is LIVE. Paste this URL into the frontend 'Server URL' box:")
    print("     ", url)
    print("  (health check: " + str(url) + "/health )")
    print("=" * 72)

if TUNNEL == "cloudflare":
    BIN = "./cloudflared"
    if not os.path.exists(BIN):
        urllib.request.urlretrieve(
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", BIN)
        os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)
    proc = subprocess.Popen([BIN, "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in proc.stdout:                       # read cloudflared logs until the URL appears
        print(line, end="")
        m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0); break
    _announce(url or "(URL not found — see the cloudflared log above)")
else:  # ngrok
    from pyngrok import ngrok
    if os.environ.get("NGROK_AUTH_TOKEN"):
        ngrok.set_auth_token(os.environ["NGROK_AUTH_TOKEN"])
    ngrok.kill()                                   # clear any tunnel left from a previous run
    public = ngrok.connect(PORT, "http")
    _announce(getattr(public, "public_url", str(public)))

app.run(port=PORT, threaded=True)
